# 06 – Markov-modellek és Dynamic HAZOP

**6. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- DTMC és CTMC megkülönböztetése
- generator matrix létrehozása és ellenőrzése
- failure–repair CTMC kiértékelése
- TK-101 állapotmodell és Dynamic HAZOP összekötése

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. Mi hiányzik a statikus FTA-ból?

A statikus fault tree nem mondja meg természetesen az eseménysorrendet, a degraded állapotban töltött időt vagy a javítás dinamikáját. Markov-modellben **állapotok** és **átmeneti intenzitások/valószínűségek** írják le az időbeli viselkedést.


## 2. DTMC

Két állapot: OK és FAILED. Egy lépéshez $P=\begin{bmatrix}.98&.02\\.20&.80\end{bmatrix}$. Sorösszeg = 1.


In [ ]:
P=np.array([[.98,.02],[.20,.80]])
p=np.array([1.,0.]); hist=[p.copy()]
for _ in range(20): p=p@P; hist.append(p.copy())
hist=np.array(hist)
plt.plot(hist[:,0],label='OK'); plt.plot(hist[:,1],label='FAILED'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 3. CTMC és generator matrix

Kétállapotú failure–repair modell:

$$Q=\begin{bmatrix}-\lambda&\lambda\\\mu&-\mu\end{bmatrix},\qquad p(t)=p(0)e^{Qt}.$$

Off-diagonal elemek nemnegatívak, minden sor összege 0.


In [ ]:
from scipy.linalg import expm
lam=.002; mu=.08
Q=np.array([[-lam,lam],[mu,-mu]])
p0=np.array([1.,0.]); times=np.linspace(0,200,200)
probs=np.vstack([p0@expm(Q*t) for t in times])
plt.plot(times,probs[:,0],label='OK'); plt.plot(times,probs[:,1],label='FAILED'); plt.axhline(mu/(lam+mu),ls='--',label='A∞'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 4. TK-101 oktatási állapotok

S0 Normal; S1 Sensor degraded; S2 Protection degraded/maintenance overdue; S3 High-level challenge; S4 Overflow; S5 Fire. Nem bontunk ki minden barrier-kombinációt külön state-ként, mert gyors state-space explosionhoz vezetne.


In [ ]:
states=['S0_normal','S1_sensor_degraded','S2_protection_degraded','S3_challenge','S4_overflow','S5_fire']; n=len(states)
Q=np.zeros((n,n))
def rate(i,j,x): Q[i,j]+=x
rate(0,1,.002); rate(0,2,.001); rate(0,3,.004)
rate(1,0,.08); rate(1,3,.004); rate(2,0,.05); rate(2,3,.004)
rate(3,0,.20); rate(3,4,.01); rate(4,0,.10); rate(4,5,.03)
for i in range(n): Q[i,i]=-Q[i].sum()
pd.DataFrame(Q,index=states,columns=states)


In [ ]:
p0=np.array([1,0,0,0,0,0.],float); ts=np.linspace(0,300,301); P=np.vstack([p0@expm(Q*t) for t in ts])
for j,s in enumerate(states): plt.plot(ts,P[:,j],label=s)
plt.xlabel('h'); plt.ylabel('state probability'); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()


## 5. Dynamic HAZOP

A HAZOP `MORE LEVEL` eltérése dinamikus útvonalként értelmezhető: **normal/degraded → high-level challenge → overflow → fire**. A safeguard vagy maintenance visszavezető átmenetet hozhat létre. Dynamic HAZOP itt nem pusztán új táblázat, hanem a deviation időbeli fejlődésének modellezése.


In [ ]:
pd.DataFrame([
 {'deviation':'MORE LEVEL','from':'S0/S1/S2','to':'S3','barrier':'alarm/trip/operator'},
 {'deviation':'NO EFFECTIVE SHUTDOWN','from':'S3','to':'S4','barrier':'LSHH+XV+manual recovery'},
 {'deviation':'IGNITION AFTER RELEASE','from':'S4','to':'S5','barrier':'containment+fire protection'}])


## 6. Maintenance sensitivity

Növeljük a sensor repair rate-et és számítsuk a 100 órán belüli overflow+fire állapotvalószínűséget.


In [ ]:
def with_repair(r):
    q=Q.copy(); q[1,0]=r; q[1,1]=-q[1,np.arange(n)!=1].sum(); return q
rows=[]
for r in [.02,.05,.08,.15]:
    p100=p0@expm(with_repair(r)*100); rows.append({'repair_rate':r,'P_S4_or_S5_100h':p100[4]+p100[5]})
pd.DataFrame(rows)


In [ ]:
from safetycourse.markov import validate_generator,transient_probabilities
validate_generator(Q); transient_probabilities(Q,p0,[0,10,100])


## 7. Korlátok

A Markov-tulajdonság szerint a jövő a jelenlegi állapottól függ. Ha a komponens ‘kora’ vagy a degraded állapotban töltött idő lényeges, azt state-be kell emelni vagy más modellt kell választani. Exponenciális tartózkodási idők sem minden folyamatra megfelelőek.


## Próbáld ki!

1. Növeld az S3→S4 rátát 0.03-ra.
2. Csökkentsd S3→S0 recovery-t 0.05-re.
3. Tedd S5-öt javíthatóvá S5→S0 átmenettel és indokold fizikailag.
4. Adj explicit S1+S2 combined state-et, és figyeld a state-space növekedését.


## Összefoglalás

A Markov-modell időbeli állapotdinamikát ad a kockázati elemzéshez, a Dynamic HAZOP pedig a veszélyes eltérések fejlődését kapcsolja ehhez. A következő tanegység Monte Carlo-val bizonytalanságot és ritka eseményeket vizsgál.
